In [22]:
# ----------------------------- Setup ----------------------------- #
# use external window for interactive buttons
%matplotlib tk

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.widgets import Button

# High-precision fields
prec = 100
RHP = RealField(prec)
CHP = ComplexField(prec)

# Example polynomial coefficients (replace as needed)
coeffs = [524288, 0, -2621440, 0, 5570560, 0, -6553600, 0, 4845120, 0,
          -2325120, 0, 715400, 0, -133760, 0, 14400, 0, -800, 0, 1]

# Convert coefficients to high-precision
coeffs_hp = [RHP(c) for c in coeffs]

# ----------------------------- Polynomial ----------------------------- #
R.<x> = RHP[]
f = sum(c*x^(len(coeffs_hp)-i-1) for i, c in enumerate(coeffs_hp))

# ----------------------------- Solve Roots ----------------------------- #
roots = f.roots(ring=CHP, multiplicities=False)

# ----------------------------- Polynomial Evaluation ----------------------------- #
def poly_eval(coeffs, r):
    p = CHP(0)
    for c in coeffs:
        p = p*r + c
    return p

def relative_residual(coeffs, r):
    numerator = abs(poly_eval(coeffs, r))
    denom = sum(abs(c)*abs(r)^(len(coeffs)-i-1) for i,c in enumerate(coeffs))
    return numerator/denom if denom != 0 else numerator

# ----------------------------- Print Roots ----------------------------- #
print(f"Polynomial degree: {len(coeffs_hp)-1}")
print("Roots with high precision and relative residuals:")
for i, r in enumerate(roots, 1):
    print(f"Root {i}: {r.n(prec)}  Residual: {relative_residual(coeffs_hp, r):.2e}")

# ----------------------------- Plotting ----------------------------- #
roots_np = np.array([complex(r) for r in roots])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18,6))

# ---- Complex Plane ----
ax1.axhline(0, color='gray', lw=1)
ax1.axvline(0, color='gray', lw=1)
ax1.scatter(roots_np.real, roots_np.imag, color='red', s=30)
ax1.set_title("Roots in Complex Plane")
ax1.set_xlabel("Re")
ax1.set_ylabel("Im")
ax1.grid(True)
ax1.set_aspect('auto')

# ---- Polynomial Curve ----
# Only real roots for highlighting
real_roots = [r.real for r in roots_np if abs(r.imag)<1e-12]

# x sampling range
x_vals = np.linspace(float(min(roots_np.real))-1, float(max(roots_np.real))+1, 2000)
y_vals = np.array([float(poly_eval(coeffs_hp, RHP(xx))) for xx in x_vals])

line, = ax2.plot(x_vals, y_vals, lw=1, label="f(x)")
ax2.axhline(0, color='gray', lw=1)
ax2.scatter(real_roots, [0]*len(real_roots), color='blue', s=30, label="Real Roots")
ax2.set_title("Polynomial Curve")
ax2.set_xlabel("x")
ax2.set_ylabel("f(x)")
ax2.grid(True)
ax2.legend()

# ----------------------------- Buttons for Linear/Symlog ----------------------------- #
max_abs = np.max(np.abs(y_vals))
linthresh = 1.0

def set_linear(event):
    ax2.set_yscale('linear')
    ax2.set_ylim(-max_abs*1.05, max_abs*1.05)
    fig.canvas.draw_idle()

def set_symlog(event):
    ax2.set_yscale('symlog', linthresh=linthresh, linscale=1.0)
    fig.canvas.draw_idle()

# Button positions (relative to figure)
axlinear = plt.axes([0.8, 0.02, 0.1, 0.04])
axsymlog = plt.axes([0.65, 0.02, 0.1, 0.04])
b_linear = Button(axlinear, 'Linear')
b_symlog = Button(axsymlog, 'Symlog')
b_linear.on_clicked(set_linear)
b_symlog.on_clicked(set_symlog)

plt.tight_layout()
plt.show()

Polynomial degree: 20
Roots with high precision and relative residuals:
Root 1: -0.52874021609613524265262898545  Residual: 2.41e-31
Root 2: -0.035764540077549292623681467452  Residual: 0.00e-27
Root 3: 0.035764540077549292623681467452  Residual: 0.00e-27
Root 4: 0.52874021609613524265262898545  Residual: 2.41e-31
Root 5: -1.2219396172495772625570553431 - 0.22052649544969188987476146861*I  Residual: 3.13e-31
Root 6: -1.2219396172495772625570553431 + 0.22052649544969188987476146861*I  Residual: 3.13e-31
Root 7: -0.70168825854194284692621277585 - 0.34929550329054614330832826013*I  Residual: 1.86e-31
Root 8: -0.70168825854194284692621277585 + 0.34929550329054614330832826013*I  Residual: 1.86e-31
Root 9: -0.69595594076906049666751837023 - 0.17232109731502796539830222252*I  Residual: 1.02e-31
Root 10: -0.69595594076906049666751837023 + 0.17232109731502796539830222252*I  Residual: 1.02e-31
Root 11: -0.34644198851106607121592630618 - 0.17311377213531120170846560683*I  Residual: 6.33e-32
Root 